# Evaluating the Efficacy of Convolutional Neural Networks in Detecting Criminal Evidence within SMS Communication

This artefact aims to use a CNN model to classify text message communications to find whether or not Convolutional Neural Networks are a viable technology for digital forensics investigations in the UK. In this case we are looking at classifying text messages as either Spam/Ham and seeing how effectively the model can classify between the two types of message.

# 1. Import necessary packages

There are several packages that we need to install to create our model, packages include pandas for managing dataset into a dataframe, numpy for mathematical elements, tensorflow for our CNN model and some sklearn modules for measuring the results of our training.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# 2. Make necessary changes to the dataset

Need to load the csv file into a dataframe and then remove columns that contain empty or irrelevant data. Also make sure to convert labels to binary 1/0 so that it can be used in numeric processes without losing their meaning. 

In [2]:
# Load in the dataset with the correct encoding.
dataset = pd.read_csv("spam_ham_dataset.csv", encoding="latin-1")

# Drop unnecessary columns
dataset.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

# Convert labels to binary (0 for not ham, 1 for spam)
dataset['v1'] = dataset['v1'].map({'ham': 0, 'spam': 1})

# Split into features and variables
texts = dataset['v2'].values
labels = dataset['v1'].values

dataset.head(20)

,v1,v2
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


# 3. Use a tokenizer to change text data into numeric values for model fitting.

In [3]:
# Hyperparameters
max_words = 1000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

# 4. Create the model and compile it.
This cell will also provide a summary of the model and how many parameters it's ouputting for the user to see.

In [4]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 128)          128000    
                                                                 
 conv1d (Conv1D)             (None, 96, 128)           82048     
                                                                 
 global_max_pooling1d (Globa  (None, 128)              0         
 lMaxPooling1D)                                                  
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                        

# 5. Train the model to fit the dataset
This code splits the data into test and train data and then trains it across five epochs. In my testing, I found that changing the number of epochs does little to improve the dataset, running more epochs significantly reduces model accuracy. I found similar results in changing the parameters of the train/test split. Having a 70% train 30% test split led to significantly less accurate results and having a 90/10 split led to minimal improvements, sometimes none at all.

In [13]:
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/5
140/140 [==============================] - 1s 5ms/step - loss: 5.0834e-04 - accuracy: 1.0000 - val_loss: 0.1240 - val_accuracy: 0.9857
Epoch 2/5
140/140 [==============================] - 1s 5ms/step - loss: 7.4853e-04 - accuracy: 0.9998 - val_loss: 0.1217 - val_accuracy: 0.9839
Epoch 3/5
140/140 [==============================] - 1s 5ms/step - loss: 2.0753e-04 - accuracy: 1.0000 - val_loss: 0.1256 - val_accuracy: 0.9839
Epoch 4/5
140/140 [==============================] - 1s 5ms/step - loss: 8.0388e-05 - accuracy: 1.0000 - val_loss: 0.1266 - val_accuracy: 0.9848
Epoch 5/5
140/140 [==============================] - 1s 5ms/step - loss: 8.5891e-05 - accuracy: 1.0000 - val_loss: 0.1340 - val_accuracy: 0.9857


# 6. Formally record overall accuracy of the model
This takes in the accuracy levels of all 5 epochs that the model has been run over. It also runs a test to get an accurate F1 reading of the data so that we can understand how it performs.

In [14]:
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print(f'F1: {f1:.2f}')

35/35 [==============================] - 0s 2ms/step
Accuracy: 0.9857
F1: 0.95


# Finally:
We can use our trained model and use it within a systematic application. In this case I've written a small function that takes a text variable as input and outputs whether or not the model believes it to be spam. I've included a couple of examples so that it can be seen to function properly, it returns the words 'spam' or 'not spam' depending on what the model believes it to be. Based on our limited testing, we can see that the model is about 98% accurate in its predictions.

If you were to take this in a professional setting and use this on (for example) a data set of 30k messages; even if you were to loop over each message and add spam messages to a list at O^N efficiency, it would take you 8 minutes to execute based on this very simple model.

In [9]:
def predict_spam(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    prediction = model.predict(padded, verbose=0)
    return "Spam" if prediction > 0.5 else "Not Spam"

# Examples
print(predict_spam("Win a free iphone now! Click here"))
print(predict_spam("Hey, how are you?!"))
      

Spam
Not Spam


# 'Real World' Example
As an example, here is a function that works to do exactly what we've just described. The model takes in a CSV dataset of one column of text messages and then iterates across them to apply the model and output a spam / not spam result. Once that result has been gleaned, you can then add any messages that are seen as 'spam' and add them to a list to output to the user at the end. To improve this into a more fully fleshed idea, you could take in more variables and return things like time stamp data, name of sender and what phone number that text message was sent from. Any other extra information such as mobile phone model and operating system could be included here if necessary.

Executing this final module will take time based on how many messages are needed to be processed. As stated earlier, this runs at O^N efficiency and such time to export increases exponentially with each additional message.

In [17]:
import time

# Load in the example data into a dataframe from CSV. 
# Load an output list into memory outside of for loop scope.
EXAMPLE_TEST_DATA = pd.read_csv("spam_example_test_data.csv")
spam_messages_output = []

# Start timer
start_time = time.time()

# let the user know that the model is processing the messages.
print("Processing...")

# Loop through the dataset, using our predict function to figure out whether the message is spam or not.
for index, row in EXAMPLE_TEST_DATA.iterrows():
    new_result = predict_spam(row.iloc[0])

    # If the result is spam, add it to the list of output messages.
    if new_result == "Spam":
        spam_messages_output.append(row.iloc[0])

# End timer and calculate execution time.
end_time = time.time()
elapsed_time = end_time - start_time

# Finally, return the ouput messages so that they can be referred to by investigators.
print(f"Time to execute: {elapsed_time} seconds.")
time.sleep(3)
print(f"Total count: {len(spam_messages_output)}")
print("Messages of interest are:")
print("-----------------------------------------------------")
for msg in spam_messages_output:
    print(msg)


Processing...
Time to execute: 32.381447315216064 seconds.
Total count: 91
Messages of interest are:
-----------------------------------------------------
Get your FREE tickets! Click here now!
Your prize is ready to claim! Hurry!
Your package has arrived! Claim it now!
Get your $500 voucher NOW! Limited time offer!
Unlock your FREE iPhone now! Limited stock!
Claim your FREE Amazon gift card here!
Get your FREE Amazon voucher now! Click here!
Congratulations, you've been selected for a FREE vacation! Click here.
FREE entry to a contest! Sign up now!
Unlock your FREE iPhone NOW — limited supply!
Claim your $1000 Amazon gift card today! Click here.
Congratulations, you’ve won a prize! Please click here to claim.
Are you free for a call?
Have you heard the latest news?
🔥 Limited time only: Get a FREE iPhone 14 now! Click here!
Congratulations! You've won a $500 gift card. Click to claim.
Get your FREE iPhone NOW! Supplies are limited, act fast!
Unlock exclusive deals and discounts. Join N